🎙️ 1단계: 연예인 목소리 "레퍼런스(Reference)" 파일 준비하기
모델이 연예인의 목소리 톤, 억양, 감정을 베끼기 위해서는 '기준이 되는 목소리(Reference Audio)'가 필요합니다
.
오디오 구하기: 좋아하는 연예인이 배경음악(BGM)이나 잡음 없이 아주 또렷하게 말하는 3~10초 분량의 오디오를 구합니다. (유튜브 인터뷰 영상 등에서 추출)
.wav 파일로 변환: 이 파일을 celebrity_voice.wav 같은 이름의 오디오 파일로 만듭니다.
대본 적어두기: 그 3~10초짜리 오디오 파일에서 연예인이 "정확히 뭐라고 말했는지" 토씨 하나 틀리지 않고 텍스트로 적어둡니다.

💾 2단계: 캐글에 파일 업로드하기
캐글 노트북을 열고 우측 패널의 [Data] 탭 또는 [Input] 디렉토리에 준비한 celebrity_voice.wav 파일을 직접 업로드합니다.

🚀 3단계: 보이스 클로닝 전용 코드 실행
세션을 새로 여셨으므로, 이전 대화에서 성공하셨던 [셀 1: 패키지 및 모델 다운로드 전용] 코드를 가장 먼저 실행하시고 **[Restart]**를 눌러 환경 세팅을 마쳐주세요.
그다음, 아래의 **[셀 2: 연예인 보이스 클로닝 전용 코드]**를 실행하시면 됩니다! (주석으로 표시된 3가지 설정만 회원님의 상황에 맞게 바꿔주세요.)

In [ ]:
# 🚀 [셀 2] 연예인 보이스 클로닝 전용 실행 코드
import sys
import soundfile as sf
import os
import gc
import torch
from IPython.display import Audio, display

print("▶️ [1단계] 메모리 대청소 및 경로 설정...")
!pkill -f api.py
gc.collect()
if torch.cuda.is_available(): 
    torch.cuda.empty_cache()

os.chdir('/kaggle/working/GPT-SoVITS')
if '/kaggle/working/GPT-SoVITS' not in sys.path:
    sys.path.append('/kaggle/working/GPT-SoVITS')
if '/kaggle/working/GPT-SoVITS/GPT_SoVITS' not in sys.path:
    sys.path.append('/kaggle/working/GPT-SoVITS/GPT_SoVITS')

os.makedirs("/kaggle/working/GPT-SoVITS/GPT_SoVITS/pretrained_models/fast_langdetect", exist_ok=True)

import inference_webui
from inference_webui import change_gpt_weights, change_sovits_weights, get_tts_wav, dict_language
dict_language["ko"] = "all_ko"
dict_language["한국어"] = "all_ko"

class DummyData: sampling_rate = 32000
class DummyHPS: data = DummyData()
inference_webui.hps = DummyHPS()

import text.korean
from mecab import MeCab
import mecab_ko_dic
try:
    dic_path = mecab_ko_dic.dictionary_path
except AttributeError:
    dic_path = "/usr/local/lib/python3.12/dist-packages/mecab_ko_dic/dicdir"
text.korean._g2p.mecab = MeCab(dictionary_path=dic_path)

# [백도어 패치]
if hasattr(inference_webui, 'bigvgan') and hasattr(inference_webui.bigvgan.BigVGAN, '_from_pretrained'):
    orig_from_pretrained = inference_webui.bigvgan.BigVGAN._from_pretrained.__func__
    @classmethod
    def patched_from_pretrained(cls, *args, **kwargs):
        kwargs.setdefault('proxies', None)
        kwargs.setdefault('resume_download', False)
        return orig_from_pretrained(cls, *args, **kwargs)
    inference_webui.bigvgan.BigVGAN._from_pretrained = patched_from_pretrained

REAL_GPT_PATH = "GPT_SoVITS/pretrained_models/s1v3.ckpt"
REAL_SOVITS_PATH = "GPT_SoVITS/pretrained_models/s2Gv3.pth"
change_gpt_weights(REAL_GPT_PATH)
change_sovits_weights(REAL_SOVITS_PATH, "ko", "ko") 

# =====================================================================
# 🔥 [여기를 수정하세요!] 보이스 클로닝 핵심 세팅 구역 🔥
# =====================================================================

# 1. 업로드한 연예인 3~10초 목소리 파일의 경로를 적어주세요.
target_ref_audio_path = "/kaggle/input/업로드한_폴더명/celebrity_voice.wav" 

# 2. 위 파일에서 연예인이 실제로 한 말을 텍스트로 정확히 적어주세요.
# (이 대본이 정확해야 AI가 억양을 완벽하게 분석합니다)
target_ref_prompt_text = "안녕하세요. 저는 오늘 인터뷰를 하러 온 OO입니다." 

# 3. 연예인이 나에게 해줬으면 하는 말을 적어주세요!
text_to_speak = "민예린! 오늘 하루도 정말 고생 많았어. 넌 최고야!"

# =====================================================================

print(f"🎙️ '{text_to_speak}' 클로닝 연산 중...")

try:
    generator = get_tts_wav(
        ref_wav_path=target_ref_audio_path,
        prompt_text=target_ref_prompt_text,
        prompt_language="ko",
        text=text_to_speak,
        text_language="ko"
    )
    
    sr, audio_data = next(generator)
    
    output_file = "celebrity_clone_output.wav"
    sf.write(output_file, audio_data, sr)
    print(f"✅ 연예인 보이스 클로닝 완료! ({output_file})")

    print("\n🎧 아래 재생 버튼을 눌러 연예인이 부르는 내 이름을 확인해 보세요!")
    display(Audio(output_file))

except Exception as e:
    print("\n❌ 🚨 클로닝 중 에러 발생! 상세 원인:")
    import traceback
    traceback.print_exc()

RVC (Retrieval-based Voice Conversion